In [1]:
# ============================================================
# RESNET CUTOUT + EMA — RETRAIN AND DIAGNOSE
#
# Original recipe:
#   4,815,940 trainable parameters
#   45,000 fitting / 5,000 validation
#   200 epochs, SGD + OneCycleLR
#   Label smoothing 0.10, Cutout 16, no color jitter
#   EMA + horizontal-flip averaging
#
# Only permitted competition training data.
# No pretrained weights or external datasets.
# No previous notebook outputs required.
# ============================================================

import os
import copy
import math
import time
import json
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display, FileLink


# ============================================================
# 1. SETTINGS
# ============================================================

EPOCHS = 200
BATCH = 128
EVAL_BATCH = 128

LR = 0.1
WD = 5e-4

VAL_SIZE = 5000
SEED = 42

WIDTHS = (42, 84, 168, 336)
BLOCKS = (2, 2, 2, 2)

LABEL_SMOOTHING = 0.1
CUTOUT = 16
COLOR_JITTER = 0.0

EMA_DECAY = 0.999
TTA = True

DATA_DIRECTORY = ""

assert torch.cuda.is_available(), (
    "Enable a GPU in Kaggle notebook settings."
)
assert EPOCHS >= 2
assert 0 < VAL_SIZE < 50000

dev = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

stamp = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

OUT = (
    Path("/kaggle/working")
    / f"four_stage_ema_diagnosis_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

CONFIG = {
    "experiment": "four_stage_cutout_ema_retrain_diagnose",
    "architecture": "four_stage_basic_resnet",
    "epochs": EPOCHS,
    "batch_size": BATCH,
    "learning_rate": LR,
    "weight_decay": WD,
    "validation_size": VAL_SIZE,
    "seed": SEED,
    "widths": list(WIDTHS),
    "blocks": list(BLOCKS),
    "label_smoothing": LABEL_SMOOTHING,
    "cutout": CUTOUT,
    "color_jitter": COLOR_JITTER,
    "use_ema": True,
    "ema_decay": EMA_DECAY,
    "ema_buffers": "average_floating_copy_integer",
    "ema_warmup": "min(decay, (1+n)/(10+n))",
    "horizontal_flip_averaging": TTA,
    "optimizer": "SGD",
    "momentum": 0.9,
    "nesterov": True,
    "scheduler": "OneCycleLR",
    "pct_start": 0.15,
    "pretrained": False,
}

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Output:", OUT)
print("Training from scratch. One GPU is used.")


# ============================================================
# 2. FIND COMPETITION DATA
# ============================================================

if DATA_DIRECTORY:
    ROOT = Path(DATA_DIRECTORY)

else:
    candidates = [
        path.parent
        for path in Path("/kaggle/input").rglob(
            "train_labels.csv"
        )
        if (path.parent / "test.csv").is_file()
        and (path.parent / "sample_submission.csv").is_file()
    ]

    assert len(candidates) == 1, (
        "Set DATA_DIRECTORY explicitly. "
        f"Candidates: {candidates}"
    )
    ROOT = candidates[0]

train_df = pd.read_csv(ROOT / "train_labels.csv")
test_df = pd.read_csv(ROOT / "test.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")

assert {"id", "filename", "label"} <= set(train_df.columns)
assert {"id", "filename"} <= set(test_df.columns)
assert sample.columns.tolist() == ["id", "label"]
assert len(train_df) == 50000

for frame in (train_df, test_df, sample):
    assert frame["id"].is_unique
    assert frame["id"].notna().all()

assert train_df[["filename", "label"]].notna().all().all()
assert test_df["filename"].notna().all()
assert train_df["filename"].is_unique
assert test_df["filename"].is_unique

numeric_labels = pd.to_numeric(
    train_df["label"], errors="raise"
)
assert np.equal(
    numeric_labels, np.floor(numeric_labels)
).all()

ytr = numeric_labels.to_numpy(dtype=np.int64)
assert ((ytr >= 0) & (ytr <= 9)).all()

assert len(test_df) == len(sample)
assert set(test_df["id"]) == set(sample["id"])

ordered_test = sample[["id"]].merge(
    test_df[["id", "filename"]],
    on="id",
    how="left",
    sort=False,
    validate="one_to_one",
)

assert ordered_test["filename"].notna().all()
assert ordered_test["id"].equals(sample["id"])

print("Data:", ROOT)


# ============================================================
# 3. LOAD PERMITTED TRAINING IMAGES
# ============================================================

def resolve_folder(name):
    direct = ROOT / name

    if direct.is_dir():
        return direct

    matches = [
        path
        for path in ROOT.rglob(name)
        if path.is_dir()
    ]

    assert len(matches) == 1, (
        f"Cannot uniquely resolve {name}: {matches}"
    )
    return matches[0]


def load_images(frame, folder):
    base = resolve_folder(folder)
    images = np.empty(
        (len(frame), 32, 32, 3), dtype=np.uint8
    )

    for index, filename in enumerate(
        frame["filename"].astype(str)
    ):
        path = base / filename

        if not path.is_file():
            path = ROOT / filename

        assert path.is_file(), f"Missing image: {path}"

        with Image.open(path) as image:
            pixels = np.array(
                image.convert("RGB"), copy=True
            )

        assert pixels.shape == (32, 32, 3), (
            str(path), pixels.shape
        )
        images[index] = pixels

    return images


print("Loading permitted training images...")
Xtr = load_images(train_df, "train_images")

hasher = hashlib.sha256()
hasher.update(train_df.to_csv(index=False).encode())
hasher.update(memoryview(Xtr))
training_fingerprint = hasher.hexdigest()


# ============================================================
# 4. ORIGINAL STRATIFIED SPLIT
# ============================================================

tr_idx, va_idx = train_test_split(
    np.arange(len(ytr)),
    test_size=VAL_SIZE,
    stratify=ytr,
    random_state=SEED,
)

assert len(np.intersect1d(tr_idx, va_idx)) == 0

np.savez(
    OUT / "data_split.npz",
    training=tr_idx,
    validation=va_idx,
)

print("Training:", len(tr_idx))
print("Validation:", len(va_idx))


# ============================================================
# 5. EXACT-DUPLICATE DIAGNOSTIC
#
# Preserve the original split for baseline reproduction.
# Report overlap instead of silently changing the experiment.
# ============================================================

image_hashes = [
    hashlib.sha256(image.tobytes()).digest()
    for image in Xtr
]

training_hashes = {
    image_hashes[index] for index in tr_idx
}

duplicate_overlap = np.array([
    image_hashes[index] in training_hashes
    for index in va_idx
])

hash_labels = {}

for image_hash, label in zip(image_hashes, ytr):
    hash_labels.setdefault(
        image_hash, set()
    ).add(int(label))

conflicting_groups = sum(
    len(values) > 1 for values in hash_labels.values()
)

print("\nDUPLICATES")
print("Unique images:", len(set(image_hashes)))
print(
    "Validation images identical to training images:",
    int(duplicate_overlap.sum()),
)
print(
    "Identical-image groups with conflicting labels:",
    conflicting_groups,
)

if duplicate_overlap.any():
    print(
        "Overlap detected. Validation accuracy may be optimistic. "
        "The original split is retained for this diagnostic run."
    )

del image_hashes, training_hashes, hash_labels


# ============================================================
# 6. NORMALIZATION — FITTING SPLIT ONLY
# ============================================================

fitting_pixels = Xtr[tr_idx]

mean_np = fitting_pixels.mean(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

std_np = fitting_pixels.std(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

del fitting_pixels

assert np.isfinite(mean_np).all()
assert np.isfinite(std_np).all()
assert (std_np > 0).all()

mean = torch.tensor(
    mean_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)

std = torch.tensor(
    std_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)


def to_gpu_uint8(images):
    return (
        torch.from_numpy(images)
        .permute(0, 3, 1, 2)
        .contiguous()
        .to(dev)
    )


Xtr_g = to_gpu_uint8(Xtr)
ytr_g = torch.from_numpy(ytr).to(dev)
tr_g = torch.as_tensor(tr_idx, device=dev)
va_g = torch.as_tensor(va_idx, device=dev)

print("\nMean:", mean_np)
print("Std:", std_np)

del Xtr


# ============================================================
# 7. ORIGINAL GPU AUGMENTATION
# ============================================================

def normalize(pixels_uint8):
    return (
        pixels_uint8.float() / 255.0 - mean
    ) / std


def augment(pixels_uint8):
    pixels = pixels_uint8.float() / 255.0
    batch_size = len(pixels)

    if COLOR_JITTER > 0:
        brightness = (
            torch.rand(
                batch_size, 1, 1, 1, device=dev
            ) * 2 - 1
        ) * COLOR_JITTER

        contrast = 1 + (
            torch.rand(
                batch_size, 1, 1, 1, device=dev
            ) * 2 - 1
        ) * COLOR_JITTER * 2

        pixels = (
            (pixels - 0.5) * contrast
            + 0.5
            + brightness
        ).clamp(0, 1)

    inputs = (pixels - mean) / std

    flip = torch.rand(batch_size, device=dev) < 0.5
    inputs = torch.where(
        flip[:, None, None, None],
        inputs.flip(-1),
        inputs,
    )

    padded = F.pad(
        inputs, (4, 4, 4, 4), mode="reflect"
    )
    axis = torch.arange(32, device=dev)

    rows = (
        torch.randint(
            0, 9, (batch_size, 1), device=dev
        ) + axis
    )
    columns = (
        torch.randint(
            0, 9, (batch_size, 1), device=dev
        ) + axis
    )

    inputs = padded[
        torch.arange(
            batch_size, device=dev
        )[:, None, None, None],
        torch.arange(
            3, device=dev
        )[None, :, None, None],
        rows[:, None, :, None],
        columns[:, None, None, :],
    ]

    if CUTOUT > 0:
        cy = torch.randint(
            0, 32, (batch_size, 1, 1), device=dev
        )
        cx = torch.randint(
            0, 32, (batch_size, 1, 1), device=dev
        )

        yy = axis.view(1, 32, 1)
        xx = axis.view(1, 1, 32)
        half = CUTOUT // 2

        mask = (
            (yy >= cy - half)
            & (yy < cy - half + CUTOUT)
            & (xx >= cx - half)
            & (xx < cx - half + CUTOUT)
        )

        inputs = inputs.masked_fill(
            mask.unsqueeze(1), 0
        )

    return inputs.contiguous(
        memory_format=torch.channels_last
    )


# ============================================================
# 8. ORIGINAL FOUR-STAGE RESNET
# ============================================================

class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()

        self.c1 = nn.Conv2d(
            cin, cout, 3, stride, 1, bias=False
        )
        self.b1 = nn.BatchNorm2d(cout)

        self.c2 = nn.Conv2d(
            cout, cout, 3, 1, 1, bias=False
        )
        self.b2 = nn.BatchNorm2d(cout)

        self.sc = None

        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(
                nn.Conv2d(
                    cin, cout, 1, stride, bias=False
                ),
                nn.BatchNorm2d(cout),
            )

    def forward(self, inputs):
        residual = F.relu(
            self.b1(self.c1(inputs))
        )
        residual = self.b2(self.c2(residual))

        shortcut = (
            inputs
            if self.sc is None
            else self.sc(inputs)
        )

        return F.relu(residual + shortcut)


class ResNet(nn.Module):
    def __init__(self, widths, blocks, nc=10):
        super().__init__()

        assert len(widths) == len(blocks)

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, widths[0], 3, 1, 1, bias=False
            ),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(),
        )

        layers = []
        cin = widths[0]

        for stage, (width, count) in enumerate(
            zip(widths, blocks)
        ):
            for block_index in range(count):
                stride = (
                    2
                    if stage > 0 and block_index == 0
                    else 1
                )

                layers.append(
                    Block(cin, width, stride)
                )
                cin = width

        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, nc)

    def forward(self, inputs):
        features = self.layers(
            self.stem(inputs)
        )
        features = F.adaptive_avg_pool2d(
            features, 1
        ).flatten(1)

        return self.fc(features)


model = (
    ResNet(WIDTHS, BLOCKS)
    .to(dev)
    .to(memory_format=torch.channels_last)
)

n_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert n_params == 4_815_940
assert n_params <= 5_000_000

print("\nTrainable parameters:", f"{n_params:,}")

# Preserve the original initial shape check.
model.eval()

with torch.inference_mode():
    assert model(
        torch.zeros(2, 3, 32, 32, device=dev)
    ).shape == (2, 10)


# ============================================================
# 9. ORIGINAL EMA
# ============================================================

class EMA:
    def __init__(self, source, decay):
        self.module = copy.deepcopy(source).eval()
        self.module.requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, source):
        self.updates += 1

        decay = min(
            self.decay,
            (1 + self.updates) / (10 + self.updates),
        )

        source_state = source.state_dict()

        for name, value in self.module.state_dict().items():
            source_value = source_state[name].detach()

            if value.is_floating_point():
                value.mul_(decay).add_(
                    source_value,
                    alpha=1.0 - decay,
                )
            else:
                value.copy_(source_value)


ema = EMA(model, EMA_DECAY)


# ============================================================
# 10. PREDICTION AND CHECKPOINT HELPERS
# ============================================================

@torch.inference_mode()
def predict(network, images_uint8, indices=None, tta=True):
    network.eval()
    outputs = []

    count = (
        len(images_uint8)
        if indices is None
        else len(indices)
    )

    for start in range(0, count, EVAL_BATCH):
        if indices is None:
            pixels = images_uint8[
                start:start + EVAL_BATCH
            ]
        else:
            pixels = images_uint8[
                indices[start:start + EVAL_BATCH]
            ]

        inputs = normalize(pixels).contiguous(
            memory_format=torch.channels_last
        )

        with torch.autocast(
            "cuda", dtype=torch.float16
        ):
            probabilities = (
                network(inputs).float().softmax(1)
            )

            if tta:
                flipped = (
                    network(inputs.flip(-1))
                    .float()
                    .softmax(1)
                )

                probabilities = (
                    probabilities + flipped
                ) / 2

        outputs.append(probabilities.cpu())

    return torch.cat(outputs)


def cpu_state(network):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in network.state_dict().items()
    }


def save_safely(payload, path):
    path = Path(path)
    temporary = path.with_suffix(".temporary")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def selected_checkpoint(weights, epoch, accuracy):
    return {
        "model": weights,
        "config": CONFIG,
        "parameters": n_params,
        "epoch": epoch,
        "validation_accuracy": accuracy,
        "mean": mean_np.tolist(),
        "std": std_np.tolist(),
        "training_fingerprint": training_fingerprint,
        "training_indices": tr_idx,
        "validation_indices": va_idx,
        "weight_source": "ema",
    }


(OUT / "config.json").write_text(
    json.dumps({
        **CONFIG,
        "parameters": n_params,
        "mean": mean_np.tolist(),
        "std": std_np.tolist(),
        "training_fingerprint": training_fingerprint,
        "torch_version": str(torch.__version__),
    }, indent=2)
)


# ============================================================
# 11. ORIGINAL OPTIMIZER AND SCHEDULER
# ============================================================

decay_parameters = []
no_decay_parameters = []

for parameter in model.parameters():
    if parameter.ndim <= 1:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

optimizer = torch.optim.SGD(
    [
        {
            "params": decay_parameters,
            "weight_decay": WD,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=LR,
    momentum=0.9,
    nesterov=True,
)

steps_per_epoch = math.ceil(len(tr_idx) / BATCH)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=EPOCHS * steps_per_epoch,
    pct_start=0.15,
)

scaler = torch.amp.GradScaler("cuda")

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ============================================================
# 12. TRAIN FROM SCRATCH
# ============================================================

best_accuracy = -1.0
best_epoch = 0
best_weights = None
history = []

selected_path = OUT / "best.pt"

print("\nSTARTING TRAINING")

for epoch in range(EPOCHS):
    started = time.time()
    model.train()

    shuffled_indices = tr_g[
        torch.randperm(len(tr_g), device=dev)
    ]

    total_loss = 0.0
    total_examples = 0
    skipped_updates = 0

    for start in range(
        0, len(shuffled_indices), BATCH
    ):
        batch_indices = shuffled_indices[
            start:start + BATCH
        ]

        inputs = augment(Xtr_g[batch_indices])
        targets = ytr_g[batch_indices]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            "cuda", dtype=torch.float16
        ):
            loss = criterion(
                model(inputs), targets
            )

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                "Nonfinite training loss."
            )

        previous_scale = scaler.get_scale()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        update_succeeded = (
            scaler.get_scale() >= previous_scale
        )

        if update_succeeded:
            scheduler.step()
            ema.update(model)
        else:
            skipped_updates += 1

        total_loss += loss.item() * len(targets)
        total_examples += len(targets)

    validation_probabilities = predict(
        ema.module,
        Xtr_g,
        indices=va_g,
        tta=TTA,
    )

    validation_accuracy = float(
        (
            validation_probabilities.argmax(1).numpy()
            == ytr[va_idx]
        ).mean()
    )

    if validation_accuracy > best_accuracy:
        best_accuracy = validation_accuracy
        best_epoch = epoch + 1
        best_weights = cpu_state(ema.module)

        save_safely(
            selected_checkpoint(
                best_weights,
                best_epoch,
                best_accuracy,
            ),
            selected_path,
        )

    seconds = time.time() - started

    history.append({
        "epoch": epoch + 1,
        "training_loss": total_loss / total_examples,
        "validation_accuracy": validation_accuracy,
        "lr_next_step": optimizer.param_groups[0]["lr"],
        "ema_updates": ema.updates,
        "skipped_amp_updates": skipped_updates,
        "seconds": seconds,
    })

    pd.DataFrame(history).to_csv(
        OUT / "training_history.csv",
        index=False,
    )

    # Recovery state saved after every completed epoch.
    save_safely({
        "model": model.state_dict(),
        "ema_model": ema.module.state_dict(),
        "ema_updates": ema.updates,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "completed_epochs": epoch + 1,
        "best_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "best_weights": best_weights,
        "history": history,
        "config": CONFIG,
        "training_fingerprint": training_fingerprint,
        "training_indices": tr_idx,
        "validation_indices": va_idx,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }, OUT / "last.pt")

    remaining = (
        np.mean([
            item["seconds"]
            for item in history[-10:]
        ])
        * (EPOCHS - epoch - 1)
        / 3600
    )

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS} | "
        f"Loss {total_loss / total_examples:.4f} | "
        f"EMA Val {validation_accuracy:.2%} | "
        f"Best {best_accuracy:.2%} "
        f"(ep {best_epoch}) | "
        f"{seconds:.0f}s | "
        f"Remaining ~{remaining:.2f}h",
        flush=True,
    )


# ============================================================
# 13. KEEP FINAL WEIGHTS FOR DIAGNOSTIC COMPARISON
# ============================================================

final_raw_weights = cpu_state(model)
final_ema_weights = cpu_state(ema.module)

# Load only your own trusted checkpoint.
checkpoint = torch.load(
    selected_path,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint["config"] == CONFIG
assert (
    checkpoint["training_fingerprint"]
    == training_fingerprint
)

del ema
optimizer.zero_grad(set_to_none=True)


# ============================================================
# 14. RAW / EMA / FLIP DIAGNOSTICS
#
# Keep original best-EMA selection for submission.
# These comparisons do not change the selected submission model.
# ============================================================

diagnostic_candidates = [
    (
        f"Selected EMA epoch {checkpoint['epoch']}",
        checkpoint["model"],
        True,
    ),
    (
        f"Final raw epoch {EPOCHS}",
        final_raw_weights,
        False,
    ),
    (
        f"Final EMA epoch {EPOCHS}",
        final_ema_weights,
        False,
    ),
]

diagnostic_rows = []
selected_val_probs = None

print("\nRUNNING CHECKPOINT DIAGNOSTICS")

for name, weights, is_selected in diagnostic_candidates:
    model.load_state_dict(weights)

    for use_flip in (False, True):
        probabilities = predict(
            model,
            Xtr_g,
            indices=va_g,
            tta=use_flip,
        ).numpy()

        val_predictions = probabilities.argmax(axis=1)
        correct = val_predictions == ytr[va_idx]

        nll = float(
            -np.log(np.clip(
                probabilities[
                    np.arange(len(va_idx)),
                    ytr[va_idx],
                ],
                1e-12,
                1.0,
            )).mean()
        )

        diagnostic_rows.append({
            "checkpoint": name,
            "inference": (
                "Flip average" if use_flip else "No flip"
            ),
            "correct": int(correct.sum()),
            "validation_accuracy": float(correct.mean()),
            "accuracy_without_exact_overlap": (
                float(correct[~duplicate_overlap].mean())
                if (~duplicate_overlap).any()
                else None
            ),
            "mean_confidence": float(
                probabilities.max(axis=1).mean()
            ),
            "negative_log_likelihood": nll,
        })

        if is_selected and use_flip == TTA:
            selected_val_probs = probabilities.copy()

comparison = pd.DataFrame(diagnostic_rows)

comparison.to_csv(
    OUT / "diagnostic_comparison.csv",
    index=False,
)

print("\nCHECKPOINT COMPARISON")
print(comparison.to_string(
    index=False,
    formatters={
        "validation_accuracy": "{:.2%}".format,
        "accuracy_without_exact_overlap": "{:.2%}".format,
        "mean_confidence": "{:.3f}".format,
        "negative_log_likelihood": "{:.4f}".format,
    },
))


# ============================================================
# 15. RESTORE ORIGINAL SELECTED MODEL AND VERIFY
# ============================================================

model.load_state_dict(checkpoint["model"])
model.eval()

assert selected_val_probs is not None

val_predictions = selected_val_probs.argmax(axis=1)
val_targets = ytr[va_idx]

verified_accuracy = float(
    (val_predictions == val_targets).mean()
)

accuracy_difference = (
    verified_accuracy
    - checkpoint["validation_accuracy"]
)

print("\nREPRODUCIBILITY")
print(
    "Recorded accuracy:",
    f"{checkpoint['validation_accuracy']:.2%}",
)
print(
    "Recomputed accuracy:",
    f"{verified_accuracy:.2%}",
)
print(
    "Difference:",
    f"{accuracy_difference * 100:+.4f} percentage points",
)

assert abs(accuracy_difference) < 1e-8, (
    "Reloaded checkpoint did not reproduce recorded accuracy."
)

print("Checkpoint accuracy reproduction: PASSED")

np.save(
    OUT / "val_probs.npy",
    selected_val_probs,
)
np.save(
    OUT / "val_ids.npy",
    train_df.iloc[va_idx]["id"].to_numpy(),
)

pd.DataFrame({
    "id": train_df.iloc[va_idx]["id"].to_numpy(),
    "true_label": val_targets,
    "prediction": val_predictions,
    "exact_overlap_with_training": duplicate_overlap,
}).to_csv(
    OUT / "validation_predictions.csv",
    index=False,
)


# ============================================================
# 16. PER-CLASS RESULTS AND CONFUSION MATRIX
# ============================================================

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

confusion = np.zeros((10, 10), dtype=np.int64)
np.add.at(
    confusion,
    (val_targets, val_predictions),
    1,
)

pd.DataFrame(
    confusion,
    index=class_names,
    columns=class_names,
).to_csv(OUT / "confusion_matrix.csv")

class_rows = []

for class_id, class_name in enumerate(class_names):
    mask = val_targets == class_id

    class_rows.append({
        "label": class_id,
        "class": class_name,
        "count": int(mask.sum()),
        "accuracy": float(
            (
                val_predictions[mask] == class_id
            ).mean()
        ),
    })

per_class = pd.DataFrame(class_rows)

per_class.to_csv(
    OUT / "validation_per_class.csv",
    index=False,
)

print("\nPER-CLASS ACCURACY")
print(per_class.to_string(
    index=False,
    formatters={"accuracy": "{:.2%}".format},
))

errors = sorted([
    (
        int(confusion[true_id, predicted_id]),
        class_names[true_id],
        class_names[predicted_id],
    )
    for true_id in range(10)
    for predicted_id in range(10)
    if true_id != predicted_id
    and confusion[true_id, predicted_id] > 0
], reverse=True)

print("\nMOST COMMON ERRORS")
for count, true_name, predicted_name in errors[:10]:
    print(
        f"{true_name} -> {predicted_name}: {count}"
    )


# ============================================================
# 17. EVALUATION INFERENCE
#
# Evaluation images are loaded only for final prediction.
# Training normalization and selected model remain fixed.
# ============================================================

del Xtr_g, ytr_g, tr_g, va_g
del final_raw_weights, final_ema_weights
del diagnostic_candidates
torch.cuda.empty_cache()

print("\nLoading evaluation images for inference...")

Xte = load_images(ordered_test, "test_images")
Xte_g = to_gpu_uint8(Xte)
del Xte

test_probabilities = predict(
    model,
    Xte_g,
    tta=TTA,
).numpy()

assert test_probabilities.shape == (
    len(ordered_test), 10
)
assert np.isfinite(test_probabilities).all()
assert np.allclose(
    test_probabilities.sum(axis=1),
    1.0,
    atol=1e-5,
)

predictions = test_probabilities.argmax(
    axis=1
).astype(np.int64)

np.save(
    OUT / "test_probs.npy",
    test_probabilities,
)
np.save(
    OUT / "test_ids.npy",
    ordered_test["id"].to_numpy(),
)


# ============================================================
# 18. WRITE AND VERIFY SUBMISSION
# ============================================================

submission = sample[["id"]].copy()
submission["label"] = predictions

assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(test_df)
assert submission["id"].equals(sample["id"])
assert submission["id"].is_unique
assert submission.notna().all().all()
assert submission["label"].between(0, 9).all()

submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)

written = pd.read_csv(submission_path)

assert written.columns.tolist() == ["id", "label"]
assert written["id"].equals(sample["id"])
assert np.array_equal(
    written["label"].to_numpy(),
    predictions,
)

# Independently verify saved probabilities against written CSV.
saved_probs = np.load(OUT / "test_probs.npy")
saved_ids = np.load(
    OUT / "test_ids.npy",
    allow_pickle=False,
)

prediction_lookup = pd.Series(
    saved_probs.argmax(axis=1),
    index=saved_ids,
)

expected = written["id"].map(prediction_lookup)

assert expected.notna().all()
assert np.array_equal(
    expected.to_numpy(),
    written["label"].to_numpy(),
)

submission_sha = hashlib.sha256(
    submission_path.read_bytes()
).hexdigest()

print("\nSUBMISSION CHECK")
print("Format and ID order: PASSED")
print("Matches saved probabilities: PASSED")
print("SHA256:", submission_sha)


# ============================================================
# 19. SAVE COMPLETE RESULTS
# ============================================================

results = {
    "experiment": CONFIG["experiment"],
    "parameters": n_params,
    "selected_epoch": int(checkpoint["epoch"]),
    "validation_accuracy": verified_accuracy,
    "training_images": len(tr_idx),
    "validation_images": len(va_idx),
    "weight_source": "ema",
    "horizontal_flip_averaging": TTA,
    "submission_rows": len(submission),
    "submission_sha256": submission_sha,
}

(OUT / "results.json").write_text(
    json.dumps(results, indent=2)
)

diagnostic_report = {
    **results,
    "exact_duplicate_validation_overlap": int(
        duplicate_overlap.sum()
    ),
    "conflicting_duplicate_groups": int(
        conflicting_groups
    ),
    "checkpoint_accuracy_difference": float(
        accuracy_difference
    ),
    "submission_matches_saved_probabilities": True,
    "checkpoint_comparison": diagnostic_rows,
    "per_class": class_rows,
}

report_path = OUT / "diagnostic_report.json"

report_path.write_text(
    json.dumps(diagnostic_report, indent=2)
)

print("\nCOMPLETE")
print("Selected checkpoint:", selected_path)
print("Recovery checkpoint:", OUT / "last.pt")
print("Submission:", submission_path)
print("Diagnostic report:", report_path)
print(
    "Predicted class counts:",
    np.bincount(predictions, minlength=10),
)
print(submission.head())

print(
    "\nDiagnostics cannot establish the cause of a lower "
    "hidden-set score by themselves."
)

display(FileLink(str(submission_path)))
display(FileLink(str(report_path)))
display(FileLink(str(OUT / "diagnostic_comparison.csv")))

GPU: Tesla T4
PyTorch: 2.11.0+cu130
Output: /kaggle/working/four_stage_ema_diagnosis_20261102_203353_214719
Training from scratch. One GPU is used.
Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Loading permitted training images...
Training: 45000
Validation: 5000

DUPLICATES
Unique images: 50000
Validation images identical to training images: 0
Identical-image groups with conflicting labels: 0

Mean: [0.49113735 0.48211119 0.44644899]
Std: [0.24686914 0.24343013 0.26160339]

Trainable parameters: 4,815,940

STARTING TRAINING
Epoch 001/200 | Loss 1.6998 | EMA Val 63.16% | Best 63.16% (ep 1) | 18s | Remaining ~0.98h
Epoch 002/200 | Loss 1.4051 | EMA Val 72.60% | Best 72.60% (ep 2) | 17s | Remaining ~0.94h
Epoch 003/200 | Loss 1.2944 | EMA Val 77.50% | Best 77.50% (ep 3) | 17s | Remaining ~0.93h
Epoch 004/200 | Loss 1.2098 | EMA Val 81.00% | Best 81.00% (ep 4) | 17s | Remaining ~0.93h
Epoch 005/200 | Loss 1.1620 | EMA Val 83.30% | Best 83.30% (ep 5) | 18s | R

/kaggle/working/four_stage_ema_diagnosis_20261102_203353_214719/submission.csv

/kaggle/working/four_stage_ema_diagnosis_20261102_203353_214719/diagnostic_report.json

/kaggle/working/four_stage_ema_diagnosis_20261102_203353_214719/diagnostic_comparison.csv